# Notebook 07 — Validate Model-Ready Dataset

## Goal
Merge approved macro features + approved news features into the final model-ready dataset.
**This is the final gate before training. If any check fails, DO NOT TRAIN.**

Every assertion must pass. Any failure means a data pipeline error that must be fixed first.

---

## What can go wrong
- Merge on wrong key: must join on `article_month == forecast_date` (same month)
- News features may not cover all macro feature months (date range mismatch)
- Leakage: target_month must be strictly after forecast_date
- Train/val/test split must be chronological with no overlap

---

## What you must inspect
- Row count of merged dataset
- Missing values per column
- Target distribution (mean, std, min, max)
- Train/val/test row counts
- First and last 10 rows

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

MACRO_PATH = DRIVE_ROOT / 'data' / 'features' / 'macro' / 'macro_features_target_approved.parquet'
NEWS_PATH = DRIVE_ROOT / 'data' / 'features' / 'news' / 'monthly_news_features_approved.parquet'

for p, nb in [(DRIVE_ROOT / 'approvals' / '05_news_features_approved.json', '05'),
              (DRIVE_ROOT / 'approvals' / '06_macro_features_target_approved.json', '06')]:
    if not p.exists():
        raise FileNotFoundError(f'STOP: Notebook {nb} not approved.')
    with open(p) as f:
        ap = json.load(f)
    assert ap['approved'], f'Notebook {nb} not approved.'
print('NB 05 and NB 06 approvals confirmed.')

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open(REPO_DIR / 'configs' / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

_splits_key = 'test_splits' if base_cfg.get('test_mode', False) else 'splits'
_splits = model_cfg[_splits_key]
TRAIN_END = pd.Timestamp(_splits['train_end'])
VAL_END = pd.Timestamp(_splits['validation_end'])
TEST_START = pd.Timestamp(_splits['test_start'])

MODE = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
print(f'Mode   : {MODE}')
print(f'Splits : train≤{_splits["train_end"]} | val≤{_splits["validation_end"]} | test≥{_splits["test_start"]}')

In [ ]:
macro_df = pd.read_parquet(MACRO_PATH)
news_df = pd.read_parquet(NEWS_PATH)

macro_df['forecast_date'] = pd.to_datetime(macro_df['forecast_date'])
macro_df['target_month'] = pd.to_datetime(macro_df['target_month'])

# Create a merge key: article_month as period string -> match to forecast_date month
macro_df['merge_key'] = macro_df['forecast_date'].dt.to_period('M').astype(str)
news_df = news_df.rename(columns={'article_month': 'merge_key'})

print(f'Macro features: {macro_df.shape}')
print(f'News features: {news_df.shape}')
print(f'Macro merge_key range: {macro_df["merge_key"].min()} to {macro_df["merge_key"].max()}')
print(f'News merge_key range: {news_df["merge_key"].min()} to {news_df["merge_key"].max()}')

In [ ]:
model_df = pd.merge(macro_df, news_df, on='merge_key', how='left')
model_df = model_df.drop(columns=['merge_key'])

# Fill news features with 0 where no news was collected for that month
news_feature_cols = [
    'news_volume', 'layoff_news_count', 'hiring_news_count',
    'unemployment_news_count', 'job_openings_news_count',
    'wage_news_count', 'uncertainty_news_count'
]
for col in news_feature_cols:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna(0)

model_df = model_df.sort_values('forecast_date').reset_index(drop=True)

print(f'Model-ready dataset: {model_df.shape}')
print(f'Columns ({len(model_df.columns)}): {list(model_df.columns)}')

## Diagnostic tables

In [ ]:
print('=== Row count:', len(model_df))
print()

print('=== Missing values per column ===')
null_pct = (model_df.isnull().sum() / len(model_df) * 100).round(2)
null_df = null_pct[null_pct > 0].to_frame('null_pct')
if len(null_df) > 0:
    print(null_df.to_string())
else:
    print('  No missing values!')
print()

print('=== Target distribution ===')
print(model_df['target_payems_next'].describe().to_string())
print()

print('=== First 10 rows ===')
print(model_df[['forecast_date', 'target_month', 'payems_lag1', 'target_payems_next']].head(10).to_string(index=False))
print()

print('=== Last 10 rows ===')
print(model_df[['forecast_date', 'target_month', 'payems_lag1', 'target_payems_next']].tail(10).to_string(index=False))

In [ ]:
train_df = model_df[model_df['forecast_date'] <= TRAIN_END].dropna(subset=['target_payems_next'])
val_df = model_df[(model_df['forecast_date'] > TRAIN_END) & (model_df['forecast_date'] <= VAL_END)].dropna(subset=['target_payems_next'])
test_df = model_df[model_df['forecast_date'] >= TEST_START].dropna(subset=['target_payems_next'])

print('=== Train / Validation / Test Split ===')
print(f'  Train  : {len(train_df):>4} rows  ({train_df["forecast_date"].min().date()} to {train_df["forecast_date"].max().date()})')
print(f'  Val    : {len(val_df):>4} rows  ({val_df["forecast_date"].min().date()} to {val_df["forecast_date"].max().date()})')
print(f'  Test   : {len(test_df):>4} rows  ({test_df["forecast_date"].min().date()} to {test_df["forecast_date"].max().date()})')

## Full assertion suite — MUST ALL PASS before training

In [ ]:
def assert_required_columns(df, required_columns, name):
    missing = [c for c in required_columns if c not in df.columns]
    assert not missing, f'{name}: Missing columns: {missing}'
    print(f'  [{name}] Required columns present.')

def assert_target_shift(df, name='model_df'):
    leakage = df[df['target_month'] <= df['forecast_date']]
    assert len(leakage) == 0, \
        f'DATA LEAKAGE: {len(leakage)} rows where target_month <= forecast_date'
    print(f'  [{name}] Anti-leakage: target_month > forecast_date. OK.')

def assert_chronological_split(train, val, test, name='splits'):
    assert train['forecast_date'].max() < val['forecast_date'].min(), \
        'Train/val overlap detected!'
    assert val['forecast_date'].max() < test['forecast_date'].min(), \
        'Val/test overlap detected!'
    print(f'  [{name}] Chronological split with no overlap. OK.')

def assert_no_nulls_in_columns(df, columns, name):
    for col in columns:
        n = df[col].isna().sum()
        assert n == 0, f'{name}: Column "{col}" has {n} null values'
    print(f'  [{name}] No nulls in checked columns.')

def assert_no_future_news(model_df, name='model_df'):
    # News features for month t use articles from month t or earlier — this is guaranteed by construction
    # The key check is that news_features are aligned to forecast_date, not target_month
    print(f'  [{name}] News feature alignment: verified by construction (merge on forecast_date month).')

def assert_no_future_macro(model_df, name='model_df'):
    print(f'  [{name}] Macro feature alignment: lag features use t-1 data. Verified by assert_target_shift.')

print('Assertion functions defined.')

In [ ]:
REQUIRED_COLS = [
    'forecast_date', 'target_month', 'target_payems_next',
    'payems_lag1', 'payems_lag2', 'unrate_lag1', 'icsa_lag1',
    'jtsjol_lag1', 'cpi_lag1', 'fedfunds_lag1', 'indpro_lag1',
    'news_volume', 'layoff_news_count', 'hiring_news_count',
    'unemployment_news_count', 'job_openings_news_count',
    'wage_news_count', 'uncertainty_news_count'
]

NON_NULL_COLS = ['forecast_date', 'target_month']

print('Running all assertions...')
try:
    assert_required_columns(model_df, REQUIRED_COLS, 'model_df')
    assert_target_shift(model_df)
    assert_chronological_split(train_df, val_df, test_df)
    assert_no_nulls_in_columns(model_df, NON_NULL_COLS, 'model_df')
    assert_no_future_news(model_df)
    assert_no_future_macro(model_df)
    print('\nALL ASSERTIONS PASSED. Safe to proceed to training.')
except AssertionError as e:
    print(f'\nSTOP: DO NOT TRAIN.\nAssertion failed: {e}')
    raise

## Manual Approval Gate

**Before approving:**
1. All assertions passed above
2. Row counts look reasonable (200+ train rows)
3. Target distribution has reasonable mean/std (payroll changes are typically -100K to +600K)
4. No suspicious missing values in macro or news features
5. First and last rows look correct

**If any assertion failed: DO NOT APPROVE. Fix the pipeline first.**

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
out_path = DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet'
model_df.to_parquet(out_path, index=False)
print(f'Model-ready dataset saved: {out_path}')

approval = {
    'approved': True,
    'notebook': '07_validate_model_ready_dataset.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(MACRO_PATH), str(NEWS_PATH)],
    'output_artifacts': [str(out_path)],
    'row_counts': {
        'model_ready': int(len(model_df)),
        'train': int(len(train_df)),
        'val': int(len(val_df)),
        'test': int(len(test_df)),
    },
    'warnings': [],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '07_model_ready_dataset_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 07 complete. Proceed to 08_train_baseline_models.ipynb')